# 05 Video Hyperparameter Tuning

Use **Optuna** to optimize hyperparameters for the CNN-LSTM deepfake detection model.

### Tuning Parameters:
- Learning Rate
- LSTM Hidden Dimension
- Dropout Rate
- Batch Size

Each trial runs for a small number of epochs on a subset of the dataset for efficiency.

In [1]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from pathlib import Path
import numpy as np
from sklearn.metrics import roc_auc_score
import os
import cv2
import json
import yaml
import logging
from PIL import Image

# Load Config
with open("../../configs/video_config.yaml", "r") as f:
    config = yaml.safe_load(f)

IMG_SIZE = config["model"]["frame_size"]
FRAME_COUNT = config["model"]["frame_count"]
TUNE_SUBSET = config["data"]["tune_subset_size"]
DATA_PATH = config["data"]["raw_path"]
TRIALS = config["tuning"]["trials"]
EPOCHS_PER_TRIAL = config["tuning"]["epochs_per_trial"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Dataset Logic

In [2]:
def discover_dataset_parts(base_path):
    base_path = Path(base_path)
    if not base_path.exists(): return []
    return sorted([base_path / d for d in os.listdir(base_path) if d.startswith("dfdc_train_part_")], key=lambda x: int(x.name.split('_')[-1]))

def get_video_list(parts_paths):
    video_list = []
    for part_path in parts_paths:
        metadata_path = part_path / "metadata.json"
        if not metadata_path.exists(): continue
        with open(metadata_path, "r") as f: metadata = json.load(f)
        for filename, info in metadata.items():
            if (part_path / filename).exists():
                video_list.append((str(part_path / filename), 1 if info["label"] == "FAKE" else 0))
    return video_list

class VideoFrameDataset(Dataset):
    def __init__(self, video_list, num_frames=32, transform=None):
        self.video_list = video_list
        self.num_frames = num_frames
        self.transform = transform
    def __len__(self): return len(self.video_list)
    def _get_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0: cap.release(); return None
        indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = Image.fromarray(frame)
                if self.transform: frame = self.transform(frame)
                frames.append(frame)
        cap.release()
        return torch.stack(frames)
    def __getitem__(self, index):
        path, label = self.video_list[index]
        frames = self._get_frames(path)
        if frames is None: frames = torch.zeros((self.num_frames, 3, IMG_SIZE, IMG_SIZE))
        return frames, torch.tensor(label, dtype=torch.float32)

## 2. Model for Tuning

In [3]:
class DeepfakeVideoModel(nn.Module):
    def __init__(self, hidden_dim, dropout):
        super(DeepfakeVideoModel, self).__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        for param in self.feature_extractor.parameters(): param.requires_grad = False
        self.lstm = nn.LSTM(input_size=1280, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1)
        )
    def forward(self, x):
        b, t, c, h, w = x.size()
        x = x.view(b * t, c, h, w)
        f = self.feature_extractor(x).view(b * t, -1).view(b, t, -1)
        _, (h_n, _) = self.lstm(f)
        return self.classifier(h_n[-1])

## 3. Optuna Objective

In [4]:
parts = discover_dataset_parts(DATA_PATH)
all_vids = get_video_list(parts)
np.random.shuffle(all_vids)
tune_subset = all_vids[:TUNE_SUBSET]
split_idx = int(0.8 * len(tune_subset))
train_vids = tune_subset[:split_idx]
val_vids = tune_subset[split_idx:]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 256, 512])
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    
    train_loader = DataLoader(VideoFrameDataset(train_vids, num_frames=FRAME_COUNT, transform=transform), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(VideoFrameDataset(val_vids, num_frames=FRAME_COUNT, transform=transform), batch_size=batch_size, shuffle=False)
    
    model = DeepfakeVideoModel(hidden_dim, dropout).to(device)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    
    for epoch in range(EPOCHS_PER_TRIAL):
        model.train()
        for f, l in train_loader:
            f, l = f.to(device), l.to(device).unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(f), l)
            loss.backward()
            optimizer.step()
            
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for f, l in val_loader:
            outputs = model(f.to(device))
            preds.extend(torch.sigmoid(outputs).cpu().numpy())
            targets.extend(l.numpy())
            
    return roc_auc_score(targets, preds)

## 4. Run Optimization

In [5]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=TRIALS)

print("Best Hyperparameters:")
print(study.best_params)

with open("best_hyperparams.json", "w") as f:
    json.dump(study.best_params, f, indent=4)